In [1]:
import pandas as pd
import numpy as np
import json
import torch
from sentence_transformers import CrossEncoder



import os
import sys
import json


# Clone Repository (Hapus folder lama biar bersih jika perlu)
if os.path.exists('skripsi-clir-code'):
    # Opsional: Paksa update
    print("Repository ditemukan. Melakukan pull...")
    !cd skripsi-clir-code && git pull
else:
    print("Cloning repository...")
    !git clone https://github.com/syifaurrr/skripsi-clir-code.git

# Tambahkan path src
REPO_PATH = './skripsi-clir-code'
TRAINING_PATH = os.path.join(REPO_PATH, 'data', 'training')
CSV_FILE_PATH = os.path.join(REPO_PATH, 'data', 'raw', 'fathul_muin.csv')  # <-- add this
sys.path.append(TRAINING_PATH)

import os
import sys
import subprocess
import torch

# --- 1a. Cek Ketersediaan GPU ---
if torch.cuda.is_available():
    print(f"✅ GPU Terdeteksi: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU tidak terdeteksi. Encoding akan berjalan di CPU (lebih lambat).")

# --- 1b. Install Library ---
print("\nMenginstall dependensi...")
libs = ["sentence-transformers", "python-terrier", "pyarabic", "faiss-cpu", "pandas", "tashaphyne"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + libs)
print("✅ Instalasi selesai.")

# --- 1c. Clone Repository ---
REPO_PATH = './skripsi-clir-code'
SRC_PATH  = os.path.join(REPO_PATH, 'src')

if os.path.exists(REPO_PATH):
    print("Repository sudah ada. Melakukan pull untuk update terbaru...")
    os.system(f"cd {REPO_PATH} && git pull")
else:
    print("Cloning repository...")
    os.system("git clone https://github.com/syifaurrr/skripsi-clir-code.git")

# Tambahkan folder src ke sys.path agar modul custom bisa di-import
if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)
print(f"✅ Path '{SRC_PATH}' ditambahkan ke sys.path.")

import re
import numpy as np
import pandas as pd
import pyterrier as pt
import pyarabic.araby as araby
from sentence_transformers import SentenceTransformer
from IPython.display import display

# Import modul preprocessing dari repository
try:
    from arabic_preprocessing import preprocess_pipeline
    print("✅ Modul arabic_preprocessing berhasil di-import.")
except ImportError as e:
    print(f"⚠️ Modul arabic_preprocessing tidak ditemukan: {e}")
    print("Pastikan struktur folder di repository benar: skripsi-clir-code/src/arabic_preprocessing.py")

# Inisialisasi PyTerrier (hanya dilakukan sekali)
if not pt.started():
    print("Menginisialisasi PyTerrier...")
    pt.init()
    print("✅ PyTerrier siap.")


# Definisikan lokasi pasti file JSON Anda
JSON_FILE_PATH = os.path.join(TRAINING_PATH, 'synthetic_query_10.json')

# --- 4a. Fungsi Normalisasi Teks Arab ---
def normalize_arabic(text):
    """Membersihkan dan menormalisasi teks Arab."""
    if not isinstance(text, str):
        return ""
    text = araby.strip_tashkeel(text)   # Hapus harakat/tanda baca
    text = araby.strip_tatweel(text)    # Hapus tatweel (pemanjang)
    text = araby.normalize_alef(text)   # Normalisasi variasi huruf Alef
    text = re.sub(r'[ى]',  'ي', text)  # Alef maqsurah -> Ya
    text = re.sub(r'[ؤ]',  'و', text)  # Waw hamza -> Waw
    text = re.sub(r'[ئ]',  'ي', text)  # Ya hamza  -> Ya
    return text

# ==========================================
# 1. LOAD DATA & FLATTENING (MEMIPIHKAN JSON)
# ==========================================
print(f"1. Membaca file JSON dari {JSON_FILE_PATH} dan mengekstrak kueri...")

# Load JSON kueri sintetik
with open(JSON_FILE_PATH, 'r', encoding='utf-8') as f:
    data_json = json.load(f)

# Load dataset asli untuk mengambil teks Arab
# Sesuaikan path ini dengan lokasi file fathul_muin.csv Anda di Kaggle
df_docs = pd.read_csv(CSV_FILE_PATH)
# normalisasi
df_docs['text_clean'] = df_docs['text'].astype(str).apply(normalize_arabic)

dict_docs = dict(zip(df_docs['docno'].astype(str), df_docs['text_clean']))
# tanpa normalisasi
# dict_docs = dict(zip(df_docs['docno'].astype(str), df_docs['text']))

flat_data = []
for item in data_json:
    pos_id = str(item.get('pos_id', ''))
    neg_id = str(item.get('neg_id', ''))
    
    pos_text = dict_docs.get(pos_id, "")
    neg_text = dict_docs.get(neg_id, "")
    
    # Ekstrak Tipe 1
    for q in item.get('tipe_1', []):
        if q.strip(): # Pastikan kueri tidak kosong
            flat_data.append({
                'query': q, 'tipe_kueri': 'tipe_1',
                'pos_id': pos_id, 'neg_id': neg_id,
                'pos_text': pos_text, 'neg_text': neg_text
            })
            
    # Ekstrak Tipe 2
    for q in item.get('tipe_2', []):
        if q.strip():
            flat_data.append({
                'query': q, 'tipe_kueri': 'tipe_2',
                'pos_id': pos_id, 'neg_id': neg_id,
                'pos_text': pos_text, 'neg_text': neg_text
            })

    # Ekstrak Tipe 3
    for q in item.get('tipe_3', []):
        if q.strip():
            flat_data.append({
                'query': q, 'tipe_kueri': 'tipe_3',
                'pos_id': pos_id, 'neg_id': neg_id,
                'pos_text': pos_text, 'neg_text': neg_text
            })

df_triplets = pd.DataFrame(flat_data)
print(f"   -> Berhasil meratakan {len(data_json)} pasangan menjadi {len(df_triplets)} baris kueri tunggal.\n")

# ==========================================
# 2. INISIALISASI CROSS-ENCODER mMARCO
# ==========================================
print("2. Memuat model Cross-Encoder mMARCO (ini memakan waktu beberapa saat)...")
# Gunakan GPU jika tersedia agar inferensi berjalan cepat
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = CrossEncoder('cross-encoder/mmarco-mMiniLMv2-L12-H384-v1', max_length=512, device=device)

# ==========================================
# 3. INFERENSI SKOR CROSS-ENCODER
# ==========================================
print("\n3. Menghitung skor kemiripan (Cross-Encoder Scoring)...")
pos_pairs = df_triplets[['query', 'pos_text']].values.tolist()
neg_pairs = df_triplets[['query', 'neg_text']].values.tolist()

print("   -> Scoring Positif F(q, p_p):")
pos_scores = model.predict(pos_pairs, batch_size=32, show_progress_bar=True)

print("   -> Scoring Negatif F(q, p_n):")
neg_scores = model.predict(neg_pairs, batch_size=32, show_progress_bar=True)

# ==========================================
# 4. PERHITUNGAN MATEMATIS (SOFTMAX MARGIN)
# ==========================================
print("\n4. Menerapkan rumus Softmax Margin...")

# Trik Numerik: Kurangi dengan nilai max untuk menghindari overflow eksponensial
max_scores = np.maximum(pos_scores, neg_scores)
exp_pos = np.exp(pos_scores - max_scores)
exp_neg = np.exp(neg_scores - max_scores)

# Hitung probabilitas Softmax
denominator = exp_pos + exp_neg
prob_pos = exp_pos / denominator
prob_neg = exp_neg / denominator

# Hitung Margin (seperti di persamaan 3.1 skripsi Anda)
margin = prob_pos - prob_neg

df_triplets['score_pos'] = pos_scores
df_triplets['score_neg'] = neg_scores
df_triplets['prob_pos'] = prob_pos
df_triplets['prob_neg'] = prob_neg
df_triplets['margin'] = margin

# ==========================================
# 5. FILTERING (MENGHAPUS HALUSINASI LLM)
# ==========================================
TAU = 0.15
df_valid = df_triplets[df_triplets['margin'] > TAU].copy()

print("\n=====================================")
print("=== HASIL AKHIR VALIDASI JH-POLO ===")
print("=====================================")
print(f"Total kueri awal (sintetis) : {len(df_triplets)} kueri")
print(f"Total kueri VALID           : {len(df_valid)} kueri")
print(f"Kueri yang dibuang (noise)  : {len(df_triplets) - len(df_valid)} kueri")
print("=====================================")

# Simpan kueri yang sudah bersih dan valid
df_valid.to_csv('triplets_valid_jhpolo.csv', index=False)
print("\n🎉 Sukses! Data latih bersih disimpan sebagai 'triplets_valid_jhpolo.csv'")

Cloning repository...
Cloning into 'skripsi-clir-code'...
remote: Enumerating objects: 1106, done.
remote: Counting objects: 100% (1106/1106), done.
remote: Compressing objects: 100% (1001/1001), done.
remote: Total 1106 (delta 172), reused 1034 (delta 100), pack-reused 0 (from 0)
Receiving objects: 100% (1106/1106), 5.80 MiB | 17.51 MiB/s, done.
Resolving deltas: 100% (172/172), done.
✅ GPU Terdeteksi: Tesla T4

Menginstall dependensi...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.5/251.5 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.8/208.8 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 69.0 MB/s eta 0:00:00
   

/usr/local/lib/python3.12/dist-packages/pyarabic/araby.py:274: SyntaxWarning: invalid escape sequence '\w'
  TOKEN_PATTERN = re.compile(u"([^\w\u0670\u064b-\u0652']+)", re.UNICODE)
/usr/local/lib/python3.12/dist-packages/pyarabic/araby.py:276: SyntaxWarning: invalid escape sequence '\w'
  TOKEN_PATTERN_SPLIT = re.compile(u"([\w\u0670\u064b-\u0652']+)", re.UNICODE)
/usr/local/lib/python3.12/dist-packages/pyarabic/araby.py:281: SyntaxWarning: invalid escape sequence '\s'
  ARABIC_STRING = re.compile(u"([^\u0600-\u0652%s%s%s\s\d])" \
/usr/local/lib/python3.12/dist-packages/pyarabic/araby.py:1237: SyntaxWarning: invalid escape sequence '\s'
  u"(?<=\s(%s|%s))%s" % (WAW, YEH, FATHA), \
/usr/local/lib/python3.12/dist-packages/pyarabic/araby.py:1450: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub(u"(?<=[\s\d])([%s])+"%(TASHKEEL_STRING),"",text,  re.UNICODE)
/tmp/ipykernel_55/623540134.py:79: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.s

terrier-assemblies 5.11 jar-with-dependencies not found, downloading to /root/.pyterrier...


https://repo1.maven.org/maven2/org/terrier/terrier-assemblies/5.11/terrier-assemblies-5.11-jar-with-dependencies.jar: 100%|██████████| 99.6M/99.6M [00:00<00:00, 166MB/s] 


Done
terrier-python-helper 0.0.8 jar not found, downloading to /root/.pyterrier...


...


3. Menghitung skor kemiripan (Cross-Encoder Scoring)...
   -> Scoring Positif F(q, p_p):


Batches:   0%|          | 0/97 [00:00<?, ?it/s]

   -> Scoring Negatif F(q, p_n):


Batches:   0%|          | 0/97 [00:00<?, ?it/s]


4. Menerapkan rumus Softmax Margin...

=== HASIL AKHIR VALIDASI JH-POLO ===
Total kueri awal (sintetis) : 3091 kueri
Total kueri VALID           : 2035 kueri
Kueri yang dibuang (noise)  : 1056 kueri

🎉 Sukses! Data latih bersih disimpan sebagai 'triplets_valid_jhpolo.csv'


In [2]:
from IPython.display import display, FileLink

# 1. Menyimpan laporan SEMUA kueri (1545 baris) beserta skornya
df_triplets.to_csv('laporan_semua_skor_jhpolo.csv', index=False)

print("✅ File siap diunduh! Silakan klik link biru di bawah ini:")
print("-" * 50)

# 2. Memunculkan tombol/link download langsung di output layar Kaggle
print(f"1. File Laporan Lengkap (Semua {len(df_triplets)} Kueri & Skor):")
display(FileLink('laporan_semua_skor_jhpolo.csv'))

print(f"\n2. File Kueri Valid ({len(df_valid)} Kueri untuk Data Latih):")
display(FileLink('triplets_valid_jhpolo.csv'))

✅ File siap diunduh! Silakan klik link biru di bawah ini:
--------------------------------------------------
1. File Laporan Lengkap (Semua 3091 Kueri & Skor):


/kaggle/working/laporan_semua_skor_jhpolo.csv


2. File Kueri Valid (2035 Kueri untuk Data Latih):


/kaggle/working/triplets_valid_jhpolo.csv